# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

My provisional lane is **Refresh / Content Opportunity Scoring**. I am choosing this lane because the starter dataset already contains page-level signals that are directly useful for a refresh decision: search visibility, clicks, CTR, average position, traffic trend, content age, last update age, sessions, engagement, and scroll behavior. For the next 7 weeks, I want to learn whether these observed signals can help rank which content pages deserve human review first, instead of treating every page as equally urgent. This is provisional: I can still confirm or change the lane by the end of Week 4 if the data shows a better direction.


In [1]:
# No code needed for this framing section.


## 2. The question: decision, action, cost of a wrong call

**Search question:** Which anonymized content pages should a content/search reviewer inspect first for refresh, expansion, metadata review, protection, pruning, or monitoring?

**Unit of analysis:** One row is one pseudonymized content item/page from the starter dataset. I will not use the IDs as model features; they are only for grouping and keeping recommendations traceable inside the anonymized dataset.

**Decision improved:** A reviewer has limited time, so the decision is not simply whether a page is "good" or "bad." The decision is which pages should be reviewed first.

**Output:** A ranked review queue at the content-page level, with a priority score, reason codes, confidence language, and a suggested action such as refresh, expand, rewrite title/meta, protect, monitor, or deprioritize.

**Action someone could take:** A content or SEO reviewer could open the highest-ranked pages first, inspect the reason codes, and decide whether to update content, improve the snippet/title/meta, add missing depth, protect a valuable page, or leave the page alone.

**Cost of a wrong recommendation:** A false positive wastes editor time and could lead someone to change a page that did not need work. A false negative means a page with real decline or opportunity may be missed. A wrong action can also be costly: for example, rewriting metadata when the real issue is seasonality, consolidation, or low search demand. Because of that, my output should support human review, not automatically prescribe changes.

**Why data or ML can help:** A single rule like "old pages need refresh" is too simple. A useful review decision depends on several signals together: demand, trend, CTR, position, age, sessions, and engagement. Data can show which combinations of signals are common among risky or promising pages. ML may help rank messy cases, but it has to beat a transparent baseline and explain its recommendations with readable reason codes. This project is not just "train a model"; it is a decision-support ranking problem.


In [2]:
# No code needed for this framing section.


## 3. Quick look at the data (2-3 real numbers)

I will use the starter CSV first because it is the dataset shipped in this repo and it is enough for a Week 1 lane choice. The numbers below are not final proof that the lane will work; they are early evidence that the refresh/opportunity question has enough real signal to investigate.


In [3]:
import pandas as pd
from pathlib import Path

candidates = [
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../../data/raw/content_refresh_anonymized.csv'),
]
data_path = next(path for path in candidates if path.exists())
df = pd.read_csv(data_path)

declining_with_demand = (df['trend_direction'].eq('down') & (df['impressions_90d'] >= 100))
low_ctr_visible = (
    (df['impressions_90d'] >= 500)
    & (df['avg_position'] > 0)
    & (df['avg_position'] <= 20)
    & (df['ctr'] < 0.5)
)
stale_visible = (
    (df['days_since_last_update'] >= 180)
    & (df['impressions_90d'] >= 500)
)

summary = pd.DataFrame(
    {
        'number': [
            len(df),
            df['client_id'].nunique(),
            int(declining_with_demand.sum()),
            int(low_ctr_visible.sum()),
            int(stale_visible.sum()),
            int(df['impressions_90d'].median()),
        ],
        'why_it_matters_for_this_lane': [
            'content pages available for starter analysis',
            'clients represented, so client-level variation matters',
            'pages are declining while still having at least 100 impressions',
            'visible pages have CTR below 0.5% while ranking in positions 1-20',
            'pages are both visible and at least 180 days since update',
            'median 90-day impressions; many rows have enough visibility to prioritize',
        ],
    },
    index=[
        'starter rows',
        'unique clients',
        'declining pages with demand',
        'low-CTR visible pages',
        'stale visible pages',
        'median impressions_90d',
    ],
)

summary


,number,why_it_matters_for_this_lane
starter rows,30000,content pages available for starter analysis
unique clients,32,"clients represented, so client-level variation..."
declining pages with demand,13152,pages are declining while still having at leas...
low-CTR visible pages,9759,visible pages have CTR below 0.5% while rankin...
stale visible pages,17,pages are both visible and at least 180 days s...
median impressions_90d,731,median 90-day impressions; many rows have enou...


## 4. Careful words: what I can and can't claim

I can claim that the starter data contains observed, anonymized search and engagement signals that may help prioritize content review. If the lane works, I can say the ranking is a **decision-support tool**: it identifies pages that look worth inspection based on measured signals such as visibility, decline, CTR, age, sessions, and engagement. I can also compare a simple baseline with a learned score using top-K metrics, because the real workflow is reviewing the first few pages in a queue.

I cannot claim that refreshing a page will cause traffic to recover, because this dataset is observational and not an experiment. I cannot claim to predict Google's algorithm, SEO rankings, or AI citations. I also cannot publish real client names, URLs, queries, domains, or titles, because the project must stay public-safe.

For later modeling, I need to be careful with labels. In the starter project, `trend_direction == "down"` is a useful beginner proxy, but it is not the same as a future outcome. If I build a stronger capstone model, I should define a prediction-time feature window and a later target window, then audit leakage before trusting the result. `trend_direction` and `trend_pct` should not become ordinary features if the label is derived from them.


In [4]:
# No code needed for this claims section.


## Self-check

Before submitting, I checked each line honestly:

- [x] Every section above is filled with markdown thinking and the code that backs the data claim.
- [x] The notebook runs top to bottom with no errors.
- [x] No client names, URLs, or private queries are included.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] The work lives under `work/notebooks/`; after committing, I can submit my repo URL on the card.
